# DBpedia SPARQL notebook

All queries use `ORDER BY` before `LIMIT` so results are deterministic and match [`dbpedia-sparql-burger.ipynb`](dbpedia-sparql-burger.ipynb).

Run SPARQL queries against the [DBpedia SPARQL endpoint](https://dbpedia.org/sparql) and show results with Polars.

| | |
|---|---|
| **Endpoint** | `https://dbpedia.org/sparql` |
| **UI** | https://dbpedia.org/sparql |

In [1]:
import time
from typing import Any
from urllib.error import HTTPError

import polars as pl
from IPython.display import display
from SPARQLWrapper import JSON, SPARQLWrapper

DBPEDIA_ENDPOINT = "https://dbpedia.org/sparql"
USER_AGENT = "SmartGridNotebook/0.1 (educational; contact: local)"

pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(-1)


def run_sparql(
    query: str,
    endpoint: str = DBPEDIA_ENDPOINT,
    retries: int = 4,
    delay_seconds: float = 3.0,
) -> pl.DataFrame:
    """Execute a SPARQL SELECT query and return bindings as a Polars DataFrame."""
    last_error: Exception | None = None

    for attempt in range(1, retries + 1):
        client = SPARQLWrapper(endpoint)
        client.setQuery(query)
        client.setReturnFormat(JSON)
        client.setTimeout(120)
        client.addCustomHttpHeader("User-Agent", USER_AGENT)

        try:
            payload: dict[str, Any] = client.query().convert()
            bindings = payload.get("results", {}).get("bindings", [])
            rows = [{key: value.get("value") for key, value in row.items()} for row in bindings]
            return pl.DataFrame(rows) if rows else pl.DataFrame()
        except HTTPError as error:
            last_error = error
            if error.code in {429, 502, 503, 504} and attempt < retries:
                print(f"DBpedia busy (HTTP {error.code}), retry {attempt}/{retries - 1}...")
                time.sleep(delay_seconds * attempt)
                continue
            raise RuntimeError(
                f"DBpedia SPARQL failed with HTTP {error.code}. "
                "The public endpoint is often overloaded — wait and retry, "
                "or paste the query at https://dbpedia.org/sparql"
            ) from error

    raise RuntimeError("DBpedia SPARQL failed after retries") from last_error


def show_df(df: pl.DataFrame) -> None:
    """Print the full table without truncating long cell values."""
    display(df)


def shorten_url_columns(df: pl.DataFrame) -> pl.DataFrame:
    """Replace full http URLs with the last path segment in each string column."""
    if df.is_empty():
        return df

    shortened = df
    for column in df.columns:
        if df[column].dtype != pl.String:
            continue
        if not df[column].str.starts_with("http").any():
            continue

        shortened = shortened.with_columns(
            pl.when(pl.col(column).str.starts_with("http"))
            .then(pl.col(column).str.split("/").list.last())
            .otherwise(pl.col(column))
            .alias(column)
        )

    return shortened


def show_last_part_of_url(df: pl.DataFrame) -> None:
    """Shorten URL columns and display the table once."""
    show_df(shorten_url_columns(df))


# Simple queries

## Query 1 — 10 distinct creator names

Without `DISTINCT`, the same creator can appear many times because resources may have multiple labels (e.g. English, German).

In [2]:
SAMPLE_QUERY = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>
SELECT DISTINCT ?creator_name WHERE {
  ?movie dbo:creator ?creator_name
}
ORDER BY ?creator_name
LIMIT 10
"""
data_frame = run_sparql(SAMPLE_QUERY) 
show_last_part_of_url(data_frame)

creator_name
str
"""%22Weird_Al%22_Yankovic"""
"""&TV"""
"""12_Yard"""
"""1984_Louisiana_World_Exposition"""
"""2waytraffic"""
"""30-Second_Bunnies_Theatre"""
"""44_Blue_Productions"""
"""4chan"""
"""5-Second_Films"""


## Query 2 — Movies whose creator label starts with "a"

In [3]:
SAMPLE_QUERY_TWO = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>
SELECT DISTINCT ?movie ?creator_name WHERE {
  ?movie dbo:creator ?creator_name . 
  ?creator_name rdfs:label ?creator_name_label .
  FILTER (STRSTARTS(LCASE(?creator_name_label), "a"))
}
ORDER BY ?movie ?creator_name
LIMIT 10
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_TWO))

movie,creator_name
str,str
"""'Sang_Linggo_nAPO_Sila""","""ABS-CBN"""
"""100_Days_to_Heaven""","""ABS-CBN_Studios"""
"""11er_Haus""","""Alfred_Dorfer"""
"""12_Hours_With""","""Aaron_Saidman"""
"""1992_(TV_series)""","""Alessandro_Fabbri_(screenwriter)"""
"""1993_(TV_series)""","""Alessandro_Fabbri_(screenwriter)"""
"""1994_(Italian_TV_series)""","""Alessandro_Fabbri_(screenwriter)"""
"""1DOL""","""ABS-CBN_Studios"""
"""2004:_The_Stupid_Version""","""Armando_Iannucci"""


## Query 3 — Distinct publication/publisher pairs (video game)

In [4]:
SAMPLE_QUERY_THREE = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>
SELECT DISTINCT ?publication ?publisher WHERE {
  ?publication dbo:publisher ?publisher;
              ?publication_type dbo:VideoGame .
}
ORDER BY ?publication ?publisher
LIMIT 5
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_THREE))

DBpedia busy (HTTP 503), retry 1/3...
DBpedia busy (HTTP 503), retry 2/3...
DBpedia busy (HTTP 503), retry 3/3...


publication,publisher
str,str
"""'90s_Super_GP""","""Nicalis"""
"""'Splosion_Man""","""Xbox_Game_Studios"""
""".detuned""","""Sony_Interactive_Entertainment"""
"""Link""","""Bandai_Namco_Entertainment"""
"""0-D_Beat_Drop""","""Aksys_Games"""


## Query 4 — Publications with English publisher name starting with "s"

In [5]:
SAMPLE_QUERY_FOUR = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>
SELECT DISTINCT ?publication ?publisher_name WHERE {
  ?publication dbo:publisher ?publisher. 
  ?publisher rdfs:label ?publisher_name .
  FILTER (lang(?publisher_name) = "en")
  FILTER (STRSTARTS(LCASE(?publisher_name), "s"))
}
ORDER BY ?publication ?publisher_name
LIMIT 5
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_FOUR))

DBpedia busy (HTTP 503), retry 1/3...
DBpedia busy (HTTP 503), retry 2/3...
DBpedia busy (HTTP 503), retry 3/3...


KeyboardInterrupt: 

## Query 5 — Video games with English publisher name starting with "a"

In [ ]:
SAMPLE_QUERY_FIVE = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>
SELECT DISTINCT ?publication ?publisher_name WHERE {
  ?publication ?publication_type dbo:VideoGame ;
               dbo:publisher ?publisher .
  ?publisher rdfs:label ?publisher_name .
  FILTER (lang(?publisher_name) = "en")
  FILTER (STRSTARTS(LCASE(?publisher_name), "a"))
}
ORDER BY ?publication ?publisher_name
LIMIT 5
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_FIVE))

publication,publisher_name
str,str
"""Jenny_of_the_Prairie""","""Addison-Wesley"""
"""Second_Extinction""","""Avalanche Studios Group"""
"""Snakebird_(video_game)""","""Astra Logical"""
"""Bejeweled_2""","""Android (operating system)"""
"""Countdown_(video_game)""","""Access Software"""


# Advanced queries

## Query 6 — Creators of video games whose English name starts with "a"

In [ ]:
SAMPLE_QUERY_SIX = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>
SELECT DISTINCT ?creator ?video_game_name WHERE {
  ?subject dbo:creator ?creator .
  ?creator ?made ?video_game . 
  ?video_game a dbo:VideoGame;
              rdfs:label ?video_game_name .
  FILTER (lang(?video_game_name) = "en")
  FILTER (STRSTARTS(LCASE(?video_game_name), "a"))
}
ORDER BY ?creator ?video_game_name
LIMIT 10
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_SIX))

creator,video_game_name
str,str
"""Kazushige_Nojima""","""Astria Ascending"""
"""List_of_Ubisoft_subsidiaries""","""Assassin's Creed III: Liberation"""
"""Leonard_Boyarsky""","""Arcanum: Of Steamworks and Magick Obscura"""
"""Tim_Cain""","""Arcanum: Of Steamworks and Magick Obscura"""
"""Brian_Mitsoda""","""Alpha Protocol"""
"""Chris_Avellone""","""Alpha Protocol"""
"""Yahtzee_Croshaw""","""Another World (video game)"""
"""Yu_Suzuki""","""After Burner"""
"""FromSoftware""","""Another Century's Episode Portable"""


## Query 7 — Video games starting with "f" that have more than one developer

In [ ]:
SAMPLE_QUERY_SEVEN = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?video_game_name (COUNT(DISTINCT ?developer) AS ?developer_count) WHERE {
  ?video_game dbo:developer ?developer ;
              a dbo:VideoGame ;
              rdfs:label ?video_game_name .
              
  FILTER (lang(?video_game_name) = "en")
  FILTER (STRSTARTS(LCASE(?video_game_name), "f"))
}
GROUP BY ?video_game_name
HAVING (COUNT(DISTINCT ?developer) > 1)
ORDER BY ?video_game_name
LIMIT 10
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_SEVEN))

video_game_name,developer_count
str,str
"""F1 Race""","""2"""
"""Fantastic Four: Rise of the Silver Surfer (video game)""","""2"""
"""FIFA 17""","""2"""
"""Fallout Online""","""2"""
"""FIFA Football 2003""","""2"""
"""Flashback (1992 video game)""","""4"""
"""Fight Night 2004""","""2"""
"""Frogger 2: Swampy's Revenge""","""2"""
"""F1 2011 (video game)""","""2"""


## Query 8 — Developers of FIFA Football 2003

In [ ]:
SAMPLE_QUERY_EIGHT = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?video_game_name ?developer WHERE {
  ?video_game a dbo:VideoGame ;
              dbo:developer ?developer ;
              rdfs:label ?video_game_name .
  FILTER (lang(?video_game_name) = "en")
  FILTER (STRSTARTS(LCASE(?video_game_name), "fifa football 2003"))
}
GROUP BY ?video_game_name
ORDER BY ?video_game_name ?developer
LIMIT 10
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_EIGHT))

video_game_name,developer
str,str
"""FIFA Football 2003""","""EA_Vancouver"""
"""FIFA Football 2003""","""Exient"""


## Query 9 — EA-published video games with release dates

In [ ]:
SAMPLE_QUERY_NINE = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?video_game_name ?publisher ?release_date WHERE {
  ?video_game a dbo:VideoGame ;
              dbo:publisher ?publisher ;
              rdfs:label ?video_game_name ;
              dbo:releaseDate ?release_date .
  ?publisher rdfs:label ?publisher_name .
  FILTER (lang(?video_game_name) = "en")
  FILTER (STRSTARTS(LCASE(?publisher_name), "ea"))
}
ORDER BY ?release_date
LIMIT 10
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_NINE))

DBpedia busy (HTTP 503), retry 1/3...
DBpedia busy (HTTP 503), retry 2/3...
DBpedia busy (HTTP 503), retry 3/3...


video_game_name,publisher,release_date
str,str,str
"""SimRefinery""","""Maxis""","""1992-10-26"""
"""SimAnt""","""Maxis""","""1993-02-26"""
"""FIFA International Soccer""","""EA_Sports""","""1993-12-03"""
"""NBA Showdown (video game)""","""EA_Sports""","""1994-03-29"""
"""NHL '94""","""EA_Sports""","""1994-03-31"""
"""FIFA Soccer 95""","""EA_Sports""","""1994-11-11"""
"""NHL 95""","""EA_Sports""","""1994-12-08"""
"""NBA Live 95""","""EA_Sports""","""1994-12-16"""
"""Full Tilt! Pinball""","""Maxis""","""1995-08-24"""


## Query 10 — EA-published video games released between 2001 and 2019

In [ ]:
SAMPLE_QUERY_TEN = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?video_game_name ?publisher ?release_date WHERE {
  ?video_game a dbo:VideoGame ;
              dbo:publisher ?publisher ;
              rdfs:label ?video_game_name ;
              dbo:releaseDate ?release_date .
              
  ?publisher rdfs:label ?publisher_name .
  
  FILTER (lang(?video_game_name) = "en")
  FILTER (STRSTARTS(LCASE(?publisher_name), "ea"))
  FILTER (YEAR(?release_date) > 2000 && YEAR(?release_date) < 2020)
}
ORDER BY ?release_date
LIMIT 10
"""
show_last_part_of_url(run_sparql(SAMPLE_QUERY_TEN))

DBpedia busy (HTTP 503), retry 1/3...
DBpedia busy (HTTP 503), retry 2/3...
DBpedia busy (HTTP 503), retry 3/3...


video_game_name,publisher,release_date
str,str,str
"""NBA Live 2001""","""EA_Sports""","""2001-01-23"""
"""NBA Live 2001""","""EA_Sports""","""2001-02-13"""
"""Triple Play Baseball""","""EA_Sports""","""2001-03-06"""
"""NHL 2001""","""EA_Sports""","""2001-03-08"""
"""Worms World Party""","""EA_Mobile""","""2001-04-06"""
"""Worms World Party""","""EA_Mobile""","""2001-04-27"""
"""NCAA Football 2002""","""EA_Sports""","""2001-07-24"""
"""FIFA Manager 09""","""EA_Sports""","""2001-09-14"""
"""FIFA Manager""","""EA_Sports""","""2001-09-14"""
